# Extended-epoch check - were the models undertrained?

**The problem.** The sweep capped training at 8 epochs with patience 5. Early stopping
needs 5 consecutive non-improving epochs to fire, so with 8 epochs it can only trigger
when the best epoch is 3 or earlier. That happened in 3 of 20 runs. The other 17 ran
until the cap and stopped because they ran out of epochs, not because they converged.

Nine runs peaked at epoch 7 or 8, and every one of the four reported configurations
contains at least one of them. So the reported means may be underestimates.

**Why it matters beyond the advisor's comment.** The Tamil result - everything stuck
in a 0.62-0.64 band, TF-IDF beating tuned XLM-R - is the evidence that model capacity
is exhausted and the next lever is in-domain data and label quality. If these models
were simply stopped too early, that conclusion is premature.

**This run.** Tamil only, both dev-selected configurations, 16 epochs, patience 5,
seed 42. At 16 epochs early stopping can fire any time the best epoch is 11 or
earlier, so convergence governs rather than the cap.

    XLM-R  lr 3e-5    8-epoch result: dev 0.6542  test 0.6295
    MuRIL  lr 1e-5    8-epoch result: dev 0.6573  test 0.6354

**Runtime:** about 60 minutes per run, 2 hours total. Resumable - each run is written
to Drive as it finishes.

In [ ]:
# ============================================================
# 0. Environment
# ============================================================
!pip -q install "transformers>=4.40" "datasets>=2.19" accelerate evaluate

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
if not torch.cuda.is_available():
    raise SystemExit("No GPU! Runtime -> Change runtime type -> T4 GPU, then rerun.")

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================
LANG   = "tamil"
EPOCHS = 16            # was 8
PATIENCE = 5           # unchanged - now it can actually fire
SEED   = 42
MAX_LEN, BATCH_SIZE, WEIGHT_DECAY = 128, 16, 0.01

# the two dev-selected configurations from the sweep, with their 8-epoch results
CONFIGS = [
    {"model": "xlm-roberta-base",       "lr": 3e-5, "old_dev": 0.6542, "old_test": 0.6295},
    {"model": "google/muril-base-cased", "lr": 1e-5, "old_dev": 0.6573, "old_test": 0.6354},
]

# multi-seed means from the sweep, for context
OLD_MEANS = {"xlm-roberta-base": (0.6164, 0.0184), "google/muril-base-cased": (0.6359, 0.0008)}
SEED_SPREAD = 0.0342
TFIDF_TAMIL = 0.630

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, gc, shutil, time
import numpy as np
import pandas as pd

BASE_DIR     = Path("/content/drive/MyDrive/hope_speech")
DATA_DIR     = BASE_DIR / "processed"
METRICS_OUT  = BASE_DIR / "metrics"
METRICS_OUT.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = METRICS_OUT / "epoch_extension_results.json"

def load_results():
    if RESULTS_PATH.exists():
        return json.loads(RESULTS_PATH.read_text())
    return {"runs": []}

def save_results(res):
    RESULTS_PATH.write_text(json.dumps(res, indent=2))

def already_done(res, model, lr):
    return any(r["model"] == model and abs(r["lr"] - lr) < 1e-12 and r["epochs"] == EPOCHS
               for r in res["runs"])

def short(m):
    return "XLM-R" if "xlm" in m else "MuRIL"

results = load_results()
print(f"Config OK - {len(results['runs'])} extended runs recorded")
print(f"Planned: {len(CONFIGS)} runs at {EPOCHS} epochs (was 8), patience {PATIENCE}")
print(f"At {EPOCHS} epochs, early stopping can fire whenever the best epoch is "
      f"{EPOCHS - PATIENCE} or earlier.")

In [ ]:
# ============================================================
# 2. Training harness - identical to the sweep except for EPOCHS
# ============================================================
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, DataCollatorWithPadding, EarlyStoppingCallback,
                          set_seed)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight

def load_split(lang, split):
    df = pd.read_csv(DATA_DIR / f"{lang}_{split}_processed.csv")
    df["text"] = df["text"].fillna("").astype(str)
    df = df[df["text"].str.strip() != ""]
    assert set(df["label"].unique()) <= {0, 1}
    return df[["text", "label"]].reset_index(drop=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro"),
            "hope_f1": f1_score(labels, preds, pos_label=1),
            "hope_precision": precision_score(labels, preds, pos_label=1, zero_division=0),
            "hope_recall": recall_score(labels, preds, pos_label=1, zero_division=0)}

_cache = {}
def get_data(lang, model_name):
    key = (lang, model_name)
    if key in _cache:
        return _cache[key]
    tok = AutoTokenizer.from_pretrained(model_name)
    dfs = {s: load_split(lang, s) for s in ("train", "dev", "test")}
    ds = {}
    for name, df in dfs.items():
        d = Dataset.from_pandas(df, preserve_index=False)
        ds[name] = d.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN),
                         batched=True, remove_columns=["text"])
    _cache[key] = (ds, dfs, tok)
    return _cache[key]

def train_one(model_name, lang, lr, seed):
    t0 = time.time()
    ds, dfs, tok = get_data(lang, model_name)
    set_seed(seed)

    cw = compute_class_weight("balanced", classes=np.array([0, 1]),
                              y=dfs["train"]["label"].values)
    cw_t = torch.tensor(cw, dtype=torch.float32).cuda()

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=cw_t)
            return (loss, outputs) if return_outputs else loss

    ckpt = Path("/content/ckpt_ext")
    shutil.rmtree(ckpt, ignore_errors=True)
    total_steps = (len(ds["train"]) / BATCH_SIZE) * EPOCHS

    args = TrainingArguments(
        output_dir=str(ckpt),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=64,
        learning_rate=lr,
        warmup_steps=int(0.1 * total_steps),
        weight_decay=WEIGHT_DECAY,
        fp16=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=500,
        seed=seed,
        report_to="none",
        disable_tqdm=True,
    )
    trainer = WeightedTrainer(
        model=model, args=args,
        train_dataset=ds["train"], eval_dataset=ds["dev"],
        data_collator=DataCollatorWithPadding(tok),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )
    trainer.train()

    hist = trainer.state.log_history
    curve      = [round(h["eval_macro_f1"], 4) for h in hist if "eval_macro_f1" in h]
    loss_curve = [round(h["eval_loss"], 4)     for h in hist if "eval_loss" in h]

    dev  = trainer.evaluate(ds["dev"])
    test = trainer.evaluate(ds["test"])
    ran  = len(curve)
    best = int(np.argmax(curve)) + 1

    rec = {"model": model_name, "lang": lang, "lr": lr, "seed": seed,
           "epochs": EPOCHS, "patience": PATIENCE,
           "epochs_ran": ran, "best_epoch": best,
           "early_stopped": ran < EPOCHS,
           "dev_macro_f1": round(dev["eval_macro_f1"], 4),
           "test_macro_f1": round(test["eval_macro_f1"], 4),
           "test_hope_f1": round(test["eval_hope_f1"], 4),
           "dev_curve": curve,
           "dev_loss_curve": loss_curve,
           "degenerate": max(curve) < 0.45,
           "minutes": round((time.time() - t0) / 60, 1)}

    del model, trainer; gc.collect(); torch.cuda.empty_cache()
    return rec

print("Harness ready.")

In [ ]:
# ============================================================
# 3. Run
# ============================================================
for cfg in CONFIGS:
    m, lr = cfg["model"], cfg["lr"]
    if already_done(results, m, lr):
        print(f"SKIP  {short(m):6s} lr={lr:g}")
        continue
    print(f"RUN   {short(m):6s} lr={lr:g} at {EPOCHS} epochs ...")
    r = train_one(m, LANG, lr, SEED)
    results["runs"].append(r); save_results(results)
    stop = "early-stopped" if r["early_stopped"] else "hit the cap again"
    print(f"      ran {r['epochs_ran']}/{EPOCHS} epochs, best at {r['best_epoch']} ({stop})")
    print(f"      dev {r['dev_macro_f1']:.4f} (was {cfg['old_dev']:.4f})  |  "
          f"test {r['test_macro_f1']:.4f} (was {cfg['old_test']:.4f})  |  {r['minutes']:.1f}min")

print("\nExtended-epoch runs complete.")

In [ ]:
# ============================================================
# 4. Verdict - was the 8-epoch cap costing anything?
# ============================================================
print("=" * 74)
print(f"EXTENDED-EPOCH CHECK - {LANG}, seed {SEED}, {EPOCHS} epochs vs the original 8")
print("=" * 74)

for cfg in CONFIGS:
    m, lr = cfg["model"], cfg["lr"]
    r = next((x for x in results["runs"]
              if x["model"] == m and abs(x["lr"] - lr) < 1e-12 and x["epochs"] == EPOCHS), None)
    if r is None:
        continue
    d_dev  = r["dev_macro_f1"]  - cfg["old_dev"]
    d_test = r["test_macro_f1"] - cfg["old_test"]
    om, osd = OLD_MEANS[m]

    print(f"\n{short(m)}  lr={lr:g}")
    print(f"  epochs run      {r['epochs_ran']}/{EPOCHS}   best epoch {r['best_epoch']}   "
          f"{'early-stopped (converged)' if r['early_stopped'] else 'hit the cap again'}")
    print(f"  dev   {cfg['old_dev']:.4f} -> {r['dev_macro_f1']:.4f}   ({d_dev:+.4f})")
    print(f"  test  {cfg['old_test']:.4f} -> {r['test_macro_f1']:.4f}   ({d_test:+.4f})")
    print(f"  3-seed mean at 8 epochs was {om:.4f} +/- {osd:.4f}")
    print(f"  dev curve: {r['dev_curve']}")
    if abs(d_test) < SEED_SPREAD:
        print(f"  -> change is inside the {SEED_SPREAD:.4f} seed spread: 8 epochs was enough")
    else:
        print(f"  -> change EXCEEDS the {SEED_SPREAD:.4f} seed spread: the cap was costing real score")

best_new = max((r["test_macro_f1"] for r in results["runs"] if r["epochs"] == EPOCHS), default=0)
print("\n" + "=" * 74)
print("WHAT THIS MEANS FOR THE TAMIL CEILING")
print("=" * 74)
print(f"  TF-IDF baseline            {TFIDF_TAMIL:.4f}")
print(f"  best 8-epoch 3-seed mean   {max(v[0] for v in OLD_MEANS.values()):.4f}")
print(f"  best {EPOCHS}-epoch single run   {best_new:.4f}")
print()
if best_new > TFIDF_TAMIL + SEED_SPREAD:
    print("  The longer run clears TF-IDF by more than the noise floor. The ceiling")
    print("  claim does NOT hold - the models were undertrained, and the sweep numbers")
    print("  understate them. Re-run the multi-seed stage before claiming anything")
    print("  about label quality.")
else:
    print("  The longer run stays inside the same band as TF-IDF and the 8-epoch")
    print("  results. The ceiling is not an artifact of the stopping rule, which")
    print("  supports moving the effort to in-domain adaptation and label quality")
    print("  rather than more training of these two models.")
print(f"\nSaved -> metrics/epoch_extension_results.json")